# Synthetic Brain MRI Master Pipeline (Colab)

This notebook runs the full 5-step pipeline from a single place.

- Step 1: preprocessing (N4, skull-strip hook, registration hook, normalization)
- Step 2: working 2D DDPM baseline (training + sampling)
- Step 3: working 3D latent diffusion-style pipeline (autoencoder + latent denoising)
- Step 4: working validation suite (proxy FID, biomarker proxy, classification utility)
- Step 5: working benchmark suite (proposed model vs DCGAN/StyleGAN2/VAE3D-style baselines)

## 1) Mount Drive (Colab)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2) Configure Paths

In [ ]:
from pathlib import Path

# TODO: change this if your repo is elsewhere on Drive
REPO_DIR = Path('/content/drive/MyDrive/AD_Research/synthetic_brain_mri')

# User-provided local Windows path (reference)
ADNI_WINDOWS_PATH = r'C:\\All-Code\\AD_Research\\ADNI1_Complete_1Yr_3T'

# Colab path to dataset root (set this to your actual mounted Drive location)
ADNI_COLAB_ROOT = Path('/content/drive/MyDrive/AD_Research/ADNI1_Complete_1Yr_3T')
ADNI_COLAB_ADNI_FOLDER = ADNI_COLAB_ROOT / 'ADNI'

RESEARCH_PLAN_LOCAL = r'C:\\All-Code\\AD_Research\\Files\\Synthetic_Brain_MRI_Generation__Research_Plan.pdf'

assert REPO_DIR.exists(), f'Repo not found: {REPO_DIR}'
assert ADNI_COLAB_ADNI_FOLDER.exists(), f'ADNI folder not found: {ADNI_COLAB_ADNI_FOLDER}'
print('Repo:', REPO_DIR)
print('ADNI folder:', ADNI_COLAB_ADNI_FOLDER)

## 3) Install Dependencies

In [ ]:
import os
os.chdir(REPO_DIR)
!pip -q install -r requirements.txt

# Optional: install HD-BET separately if you want skull stripping active.
# !pip install hd-bet

## 4) Patch Config for Colab and Run Pipeline

In [ ]:
import subprocess
import sys
import yaml

cfg_path = REPO_DIR / 'config' / 'config.yaml'
with cfg_path.open('r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

cfg['paths']['adni_root'] = str(ADNI_COLAB_ADNI_FOLDER)
cfg['paths']['data_raw'] = str(REPO_DIR / 'data' / 'raw')
cfg['paths']['data_interim'] = str(REPO_DIR / 'data' / 'interim')
cfg['paths']['data_processed'] = str(REPO_DIR / 'data' / 'processed')
cfg['paths']['step1_output'] = str(REPO_DIR / 'results' / 'step1')

# Fast debug mode on Colab first; set to False for full run
cfg['step1']['smoke_test'] = True
cfg['step1']['smoke_test_cases'] = 5
cfg['step1']['num_workers'] = 2

tmp_cfg = REPO_DIR / 'config' / 'config.colab.yaml'
with tmp_cfg.open('w', encoding='utf-8') as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

cmd = [sys.executable, 'run_all_steps.py', '--config', str(tmp_cfg), '--start-step', '1', '--end-step', '5']
print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)
print('Pipeline run completed.')

## 5) Full Run Toggle

For full Step 1 on all scans, set:
- `cfg['step1']['smoke_test'] = False`
- optionally increase `cfg['step1']['num_workers']` based on runtime RAM/CPU.